### BERT Soft-Label Reappraisal Experiment Runner
Training BERT models for reappraisal strategy classificatioin using soft annotator labels.
Target: proportion of annotators who marked each strategy present.


In [ ]:
from __future__ import annotations

import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

# display makes tables render nicely in Jupyter and keeps dry runs explicit.
from IPython.display import display

# Core scientific Python stack.
# numpy/pandas handle the annotation matrices and saved result tables.
import numpy as np
import pandas as pd

# PyTorch is used for the model, dataloaders, custom loss, and training loop.
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

# Hugging Face transformers supplies the tokenizer, pretrained encoders, and learning-rate scheduler.
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

# Metrics for soft-label evaluation and thresholded multi-label evaluation.
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    hamming_loss,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split

# If not installed the notebook uses random splits.
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except ModuleNotFoundError:
    MultilabelStratifiedShuffleSplit = None
    HAS_ITERSTRAT = False

print('iterative-stratification available:', HAS_ITERSTRAT)


Project folders, output folder, and the eight reappraisal strategy labels.


In [ ]:
PROJECT_DIR = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'data' / 'annotate-lime.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')
DATA_DIR = PROJECT_DIR / 'data'

OUTPUT_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'baseline'
SUMMARY_DIR = OUTPUT_DIR / 'summaries'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# 8 strategies
REAPPRAISAL_TECHNIQUES = ['[GM]', '[PR]', '[A]', '[EJ]', '[N]', '[UC]', '[V]', '[H]']
LABEL_COLS = [tech.strip('[]') for tech in REAPPRAISAL_TECHNIQUES]

LABEL_NAMES = {
    'GM': 'growth mindset',
    'PR': 'positive repurposing',
    'A': 'aberration',
    'EJ': 'external justification',
    'N': 'normalizing',
    'UC': 'unwarranted conclusions',
    'V': 'validation',
    'H': 'humor',
}

print('Project directory:', PROJECT_DIR)
print('Output directory:', OUTPUT_DIR)
print('Labels:', LABEL_COLS)


### Model Configurations
Different Models being used. Each ModelConfig gives the configs of experiment: the Hugging Face model name, batch size, learning rate, dropout, sequence length, early-stopping settings.

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    experiment_name: str
    model_name: str
    batch_size: int = 8
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    dropout: float = 0.4
    max_length: int = 256
    num_epochs: int = 12
    patience: int = 3
    local_files_only: bool = False


# different Bert models
MODEL_CONFIGS = [
    ModelConfig(
        experiment_name='bert_base_uncased',
        model_name='bert-base-uncased',
        batch_size=8,
        learning_rate=1e-5,
        dropout=0.4,
        local_files_only=True,
    ), # old runs showed best results for these configs, drop out quite high cuz dataset quite small

#If training loss drops but validation loss worsens,have to increase dropout. If both train and validation are bad, dropout potentially too high.


    ModelConfig(
        experiment_name='bert_large_uncased',
        model_name='google-bert/bert-large-uncased',
        batch_size=4,
        learning_rate=5e-6,
        dropout=0.5,
        local_files_only=False,
    ), # model is bigger therefore possibly overfits more easily, thus smaller config and more regularization


    ModelConfig(
        experiment_name='modernbert_base',
        model_name='answerdotai/ModernBERT-base',
        batch_size=8,
        learning_rate=1e-5,
        dropout=0.4,
        local_files_only=False,
    ), #uses same as bertbase
]


MODEL_CONFIGS


In [ ]:
# random seeds
def set_seed(seed: int) -> None:

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


#Use CUDA
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print('Using device:', device)


### Load The Reappraisal Dataset

Building the  modeling table. For every strategy it looks at the corresponding annotation & computes:

- positive votes
- number of valid votes
- soft score = positive votes / valid votes

The model sees `SITUATION` and `REAPPRAISAL` as text input. The target is the eight soft-label scores.

In [ ]:
def load_reappraisal_dataset(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    texts = pd.read_csv(data_dir / 'annotate-lime.csv')

    soft_labels = pd.DataFrame()
    vote_counts = pd.DataFrame()
    positive_votes = pd.DataFrame() # if strategy was given 1 for present

    for tech in REAPPRAISAL_TECHNIQUES:
        label = tech.strip('[]')

        # Each strategy has one semicolon-separated annotation matrix.
        ann = pd.read_csv(data_dir / f'annotations-only-{tech}.csv', sep=';')
        ann = ann.apply(pd.to_numeric, errors='coerce')

        # raw vote counts and the observed annotator proportion.
        positive_votes[f'{label}_positive_votes'] = ann.sum(axis=0).values
        vote_counts[f'{label}_n_votes'] = ann.count(axis=0).values
        soft_labels[label] = ann.mean(axis=0).values

    dataset = pd.concat(
        [
            texts[['key', 'SITUATION', 'REAPPRAISAL']].reset_index(drop=True),
            soft_labels.reset_index(drop=True),
            positive_votes.reset_index(drop=True),
            vote_counts.reset_index(drop=True),
        ],
        axis=1,
    )

    dataset['vignette_id'] = pd.factorize(dataset['SITUATION'])[0]

    # Sanity check: the soft labels should exactly equal positive_votes / n_votes.
    expected_scores = (
        dataset[[f'{label}_positive_votes' for label in LABEL_COLS]].to_numpy()
        / dataset[[f'{label}_n_votes' for label in LABEL_COLS]].to_numpy()
    )
    if not np.allclose(dataset[LABEL_COLS].to_numpy(), expected_scores, equal_nan=True):
        raise ValueError('Soft labels do not match positive_votes / n_votes.')

    return dataset


# Secondary binary reports use three interpretations of what it means for a
# strategy to be "present". Training still uses the soft labels above.
BINARY_TARGET_SCHEMES = {
    'weak_presence': 'at least one annotator marked the strategy present',
    'majority': 'more than half of valid annotators marked the strategy present',
    'strong_presence': 'at least 80% of valid annotators marked the strategy present',
}


def binary_target_matrix(df: pd.DataFrame, scheme: str = 'majority') -> np.ndarray:

    positive_votes = df[[f'{label}_positive_votes' for label in LABEL_COLS]].to_numpy()
    n_votes = df[[f'{label}_n_votes' for label in LABEL_COLS]].to_numpy()
    soft_scores = positive_votes / np.clip(n_votes, 1, None)

    if scheme == 'weak_presence':
        return (positive_votes >= 1).astype(int)
    if scheme == 'majority':
        return (positive_votes > (n_votes / 2)).astype(int)
    if scheme == 'strong_presence':
        return (soft_scores >= 0.80).astype(int)
    raise ValueError(f'Unknown binary target scheme: {scheme}')


def binary_targets(df: pd.DataFrame, scheme: str = 'majority') -> pd.DataFrame:
    #Return binary target labels as a DF with label columns
    return pd.DataFrame(binary_target_matrix(df, scheme), columns=LABEL_COLS, index=df.index)


dataset = load_reappraisal_dataset()
print('Dataset shape:', dataset.shape)
print('Number of situations:', dataset['SITUATION'].nunique())
display(dataset[['key', 'SITUATION', 'REAPPRAISAL'] + LABEL_COLS].head())

# Show how many positives each binary definition creates.
binary_count_summary = pd.concat(
    [binary_targets(dataset, scheme).sum().rename(scheme) for scheme in BINARY_TARGET_SCHEMES],
    axis=1,
)
display(binary_count_summary)


## Train/Validation/Test Splits

Iterative multilabel stratification to preserve rare labels across train, validation, and test sets. Else notebook uses seeded random split.

In [ ]:
def multilabel_split_indices(
    df: pd.DataFrame,
    y_binary: pd.DataFrame,
    test_size: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
 #Return train/test indices using multilabel stratification

    if HAS_ITERSTRAT:
        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed,
        )
        return next(splitter.split(df, y_binary))

    print(
        'Warning: iterative stratification not installed; '
        'using a seeded random split.'
    )
    indices = np.arange(len(df))
    return train_test_split(
        indices,
        test_size=test_size,
        random_state=seed,
        shuffle=True,
    )


def make_random_split(
    dataset: pd.DataFrame,
    seed: int,
    split_scheme: str = 'majority',
    test_size: float = 0.15,
    val_size: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    #70/15/15 train/validation/test split. first split creates train vs. temporary data. The second split divides the temporary data equally into validation and test.


    temp_size = test_size + val_size
    train_idx, temp_idx = multilabel_split_indices(
        dataset,
        binary_targets(dataset, split_scheme),
        test_size=temp_size,
        seed=seed,
    )

    train_df = dataset.iloc[train_idx].reset_index(drop=True)
    temp_df = dataset.iloc[temp_idx].reset_index(drop=True)

    test_fraction_of_temp = test_size / temp_size
    val_idx, test_idx = multilabel_split_indices(
        temp_df,
        binary_targets(temp_df, split_scheme),
        test_size=test_fraction_of_temp,
        seed=seed,
    )

    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)
    return train_df, val_df, test_df


def make_leave_one_vignette_splits(
    dataset: pd.DataFrame,
    seed: int,
    split_scheme: str = 'majority',
    val_size: float = 0.15,
) -> list[tuple[str, pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
    #leave one out

    splits = []
    for situation, test_df in dataset.groupby('SITUATION', sort=False):
        train_val_df = dataset.loc[dataset['SITUATION'] != situation].reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)

        train_idx, val_idx = multilabel_split_indices(
            train_val_df,
            binary_targets(train_val_df, split_scheme),
            test_size=val_size,
            seed=seed,
        )
        train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
        val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
        splits.append((situation, train_df, val_df, test_df))

    return splits


train_df, val_df, test_df = make_random_split(dataset, seed=42, split_scheme='majority')
print('Example random split sizes:', len(train_df), len(val_df), len(test_df))


### PyTorch Dataset

The dataset object turns each row into tensors.

- tokenizer outputs for SITUATION and REAPPRAISAL.
- soft labels for the eight strategy scores
- positive vote counts
- valid vote counts


In [ ]:
class ReappraisalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        # Passing situation and reappraisal as a text pair lets BERT use segment information
        # for models that support token_type_ids.
        encoding = self.tokenizer(
            str(row['SITUATION']),
            str(row['REAPPRAISAL']),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )

        # Remove the extra batch dimension added by return_tensors='pt'.
        item = {key: value.squeeze(0) for key, value in encoding.items()}

        # Soft target: observed annotator proportion for each strategy.
        item['labels'] = torch.tensor(row[LABEL_COLS].values.astype('float32'))

        # Raw annotation counts used by binary cross-entropy + sigmoid logits
        item['positive_votes'] = torch.tensor(
            row[[f'{label}_positive_votes' for label in LABEL_COLS]].values.astype('float32')
        )
        item['n_votes'] = torch.tensor(
            row[[f'{label}_n_votes' for label in LABEL_COLS]].values.astype('float32')
        )
        return item


### Model Definition

Model wraps Hugging Face encoder with a shared eight-output prediction head.

The architecture is:
Situation + Reappraisal -> pretrained encoder -> pooled representation -> dropout -> linear layer -> 8 logits

The logits become probabilities after applying sigmoid.

In [ ]:
class EncoderForReappraisalSoftLabels(nn.Module):
    def __init__(self, config: ModelConfig, num_labels: int = len(LABEL_COLS)):
        super().__init__()

        # AutoModel lets this same class load BERT, BERT-large, ModernBERT, etc.
        self.encoder = AutoModel.from_pretrained(
            config.model_name,
            local_files_only=config.local_files_only,
        )
        self.dropout = nn.Dropout(config.dropout)
        self.regressor = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, **encoder_inputs) -> torch.Tensor:
        outputs = self.encoder(**encoder_inputs)

        # BERT exposes pooler_output. Some newer encoders may not, so we fall back to CLS.
        pooled = getattr(outputs, 'pooler_output', None)
        if pooled is None:
            pooled = outputs.last_hidden_state[:, 0]

        return self.regressor(self.dropout(pooled))


### Loss Functions

The recommended loss binary cross-entropy with sigmoid logits.


k = number of annotators who marked the strategy present
& n = number of valid annotator judgments


Better than training only on the mean score because it preserves that labels came from multiple judgments.

Also possible bce_soft and mse as comparison option.

In [ ]:
def split_batch(batch: dict[str, torch.Tensor], device: torch.device):
    labels = batch['labels'].to(device)
    positive_votes = batch['positive_votes'].to(device)
    n_votes = batch['n_votes'].to(device)

    # Keep every tokenizer field returned by the tokenizer.
    # This avoids mismatches between training and prediction for models with token_type_ids.
    encoder_inputs = {
        key: value.to(device)
        for key, value in batch.items()
        if key not in {'labels', 'positive_votes', 'n_votes'}
    }
    return encoder_inputs, labels, positive_votes, n_votes


def binomial_nll_loss(logits: torch.Tensor, positive_votes: torch.Tensor, n_votes: torch.Tensor) -> torch.Tensor:
    #binary cross-entropy with sigmoid logits

    targets = positive_votes / n_votes.clamp_min(1.0)
    per_vote_bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    return (per_vote_bce * n_votes).sum() / n_votes.sum().clamp_min(1.0)


def compute_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    positive_votes: torch.Tensor,
    n_votes: torch.Tensor,
    loss_mode: str,
) -> torch.Tensor:
    #Dispatch to loss function

    if loss_mode == 'binomial_nll':
        return binomial_nll_loss(logits, positive_votes, n_votes)
    if loss_mode == 'bce_soft':
        return F.binary_cross_entropy_with_logits(logits, labels)
    if loss_mode == 'mse':
        return F.mse_loss(torch.sigmoid(logits), labels)
    raise ValueError("loss_mode must be 'binomial_nll', 'bce_soft', or 'mse'.")


### Training And Prediction Helpers

Training epoch, validation loss, and prediction.

Optimizer uses weight decay when appropriate: biases and layer-norm weights are excluded from weight decay.

In [ ]:
def build_optimizer(model: nn.Module, learning_rate: float, weight_decay: float) -> AdamW:
    #AdamW with no weight decay on bias and layer-normalization parameters

    no_decay = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    grouped_params = [
        {
            'params': [
                p for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)
            ],
            'weight_decay': weight_decay,
        },
        {
            'params': [
                p for n, p in model.named_parameters()
                if p.requires_grad and any(nd in n for nd in no_decay)
            ],
            'weight_decay': 0.0,
        },
    ]
    return AdamW(grouped_params, lr=learning_rate)


def train_one_epoch(model, dataloader, optimizer, scheduler, device, loss_mode: str) -> float:
    #Train the model for one pass over the training data

    model.train()
    total_loss = 0.0

    for batch in dataloader:
        optimizer.zero_grad(set_to_none=True)
        encoder_inputs, labels, positive_votes, n_votes = split_batch(batch, device)
        logits = model(**encoder_inputs)
        loss = compute_loss(logits, labels, positive_votes, n_votes, loss_mode)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_loss(model, dataloader, device, loss_mode: str) -> float:
    """Compute validation/test loss without updating model weights."""

    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            encoder_inputs, labels, positive_votes, n_votes = split_batch(batch, device)
            logits = model(**encoder_inputs)
            loss = compute_loss(logits, labels, positive_votes, n_votes, loss_mode)
            total_loss += loss.item()

    return total_loss / len(dataloader)


def get_predictions(model, dataloader, device) -> tuple[np.ndarray, np.ndarray]:
    #Return sigmoid probabilities and true soft labels for a dataloader

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            encoder_inputs, labels, _, _ = split_batch(batch, device)
            logits = model(**encoder_inputs)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return np.vstack(all_probs), np.vstack(all_labels)


### Evaluation Metrics
- MAE
- RMSE
- Pearson correlation
- Spearman correlation

Thresholded F1 only for understandability of binary classification.

In [ ]:
def safe_corr(fn, y_true: np.ndarray, y_score: np.ndarray) -> float:
    if np.std(y_true) == 0 or np.std(y_score) == 0:
        return np.nan
    return float(fn(y_true, y_score).statistic)


def soft_label_metrics(y_true: np.ndarray, y_score: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, label in enumerate(LABEL_COLS):
        yt = y_true[:, i]
        ys = y_score[:, i]
        rows.append({
            'label': label,
            'strategy': LABEL_NAMES[label],
            'mae': mean_absolute_error(yt, ys),
            'rmse': mean_squared_error(yt, ys) ** 0.5,
            'pearson': safe_corr(pearsonr, yt, ys),
            'spearman': safe_corr(spearmanr, yt, ys),
            'mean_true': yt.mean(),
            'mean_pred': ys.mean(),
        })
    return pd.DataFrame(rows)


def macro_soft_summary(metrics_df: pd.DataFrame) -> dict[str, float]:
    return {
        'macro_mae': float(metrics_df['mae'].mean()),
        'macro_rmse': float(metrics_df['rmse'].mean()),
        'mean_pearson': float(metrics_df['pearson'].mean(skipna=True)),
        'mean_spearman': float(metrics_df['spearman'].mean(skipna=True)),
    }


### Threshold Tuning And Binary Metrics

The model is trained on soft labels. After training also three secondary binary labels:

- weak presence: at least one annotator marked the strategy present
- majority: more than half of valid annotators marked the strategy present
- strong presence: at least 80% of valid annotators marked the strategy present

For each, prediction thresholds are tuned on the validation set and evaluated on the test set.


In [ ]:
def tune_thresholds(
    y_true_binary: np.ndarray,
    y_score: np.ndarray,
    grid: Iterable[float] | None = None,
) -> tuple[dict[str, float], pd.DataFrame]:
    #prediction threshold per label using validation F1


    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)

    thresholds = {}
    tuning_rows = []

    for i, label in enumerate(LABEL_COLS):
        # If validation contains only one class for a rare label, F1 tuning is undefined.
        # In that case, keep the neutral 0.5 threshold and record the issue.
        if len(np.unique(y_true_binary[:, i])) < 2:
            thresholds[label] = 0.5
            tuning_rows.append({
                'label': label,
                'threshold': 0.5,
                'validation_f1': np.nan,
                'note': 'validation set has one class',
            })
            continue

        best_threshold = 0.5
        best_f1 = -1.0
        for threshold in grid:
            pred = (y_score[:, i] >= threshold).astype(int)
            score = f1_score(y_true_binary[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_threshold = float(threshold)

        thresholds[label] = best_threshold
        tuning_rows.append({
            'label': label,
            'threshold': best_threshold,
            'validation_f1': best_f1,
            'note': '',
        })

    return thresholds, pd.DataFrame(tuning_rows)


def apply_thresholds(scores: np.ndarray, thresholds: dict[str, float]) -> np.ndarray:
    """Apply per-label prediction thresholds to the model score matrix."""

    threshold_array = np.array([thresholds[label] for label in LABEL_COLS])
    return (scores >= threshold_array).astype(int)


def evaluate_multilabel(
    y_true_binary: np.ndarray,
    scores: np.ndarray,
    thresholds: dict[str, float],
) -> tuple[dict[str, float], pd.DataFrame, pd.DataFrame, str, np.ndarray]:
    #Evaluate thresholded predictions as a conventional multi-label task

    y_pred = apply_thresholds(scores, thresholds)

    metrics = {
        'micro_f1': f1_score(y_true_binary, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true_binary, y_pred, average='macro', zero_division=0),
        'samples_f1': f1_score(y_true_binary, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true_binary, y_pred),
    }

    ap_rows = []
    count_rows = []
    for i, label in enumerate(LABEL_COLS):
        ap = np.nan
        if len(np.unique(y_true_binary[:, i])) >= 2:
            ap = average_precision_score(y_true_binary[:, i], scores[:, i])
        ap_rows.append({'label': label, 'average_precision': ap})
        count_rows.append({
            'label': label,
            'true_positive_items': int(y_true_binary[:, i].sum()),
            'predicted_positive_items': int(y_pred[:, i].sum()),
        })

    report = classification_report(y_true_binary, y_pred, target_names=LABEL_COLS, zero_division=0)
    return metrics, pd.DataFrame(ap_rows), pd.DataFrame(count_rows), report, y_pred


### Saving Results

 checkpoint, metrics, predictions, and summary files.


In [ ]:
def sanitize_for_json(value):
    #Convert NumPy values and NaNs into JSON values

    if isinstance(value, dict):
        return {key: sanitize_for_json(inner) for key, inner in value.items()}
    if isinstance(value, list):
        return [sanitize_for_json(inner) for inner in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if math.isnan(float(value)):
            return None
        return float(value)
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def write_json(path: Path, payload: dict) -> None:
    """Write metrics as strict JSON."""

    path.write_text(json.dumps(sanitize_for_json(payload), indent=2))


### Run One Experiment

 Train one model on one split, evaluate soft-label prediction on the test set, then report weak/majority/strong binary metrics as secondary analyses.


In [ ]:
def run_artifact_dir(config: ModelConfig, run_id: str, output_dir: Path = OUTPUT_DIR) -> Path:
    return output_dir / config.experiment_name / run_id.replace('random_', '')


def result_metrics_path(config: ModelConfig, run_id: str, output_dir: Path = OUTPUT_DIR) -> Path:
    run_dir = run_artifact_dir(config, run_id, output_dir)
    return run_dir / f'{config.experiment_name}_{run_id}_metrics.json'


def load_existing_result(config: ModelConfig, run_id: str, output_dir: Path = OUTPUT_DIR) -> dict | None:
    path = result_metrics_path(config, run_id, output_dir)
    return json.loads(path.read_text()) if path.exists() else None


def run_single_experiment(
    config: ModelConfig,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    run_id: str,
    output_dir: Path = OUTPUT_DIR,
    loss_mode: str = 'binomial_nll',
    device: torch.device | None = None,
) -> dict:
    #Train and evaluate one model configuration on one data split.

    run_dir = run_artifact_dir(config, run_id, output_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    device = device or get_device()

    # Tokenizer and dataset are created inside the run so each model uses its own tokenizer.
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name,
        local_files_only=config.local_files_only,
    )
    train_loader = DataLoader(
        ReappraisalDataset(train_df, tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        ReappraisalDataset(val_df, tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        ReappraisalDataset(test_df, tokenizer, config.max_length),
        batch_size=config.batch_size,
        shuffle=False,
    )

    model = EncoderForReappraisalSoftLabels(config).to(device)
    optimizer = build_optimizer(model, config.learning_rate, config.weight_decay)

    total_training_steps = len(train_loader) * config.num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.10 * total_training_steps),
        num_training_steps=total_training_steps,
    )

    # Checkpoint name includes both model and run id, so different experiments do not overwrite each other.
    checkpoint_path = run_dir / f'{config.experiment_name}_{run_id}_best_model.pt'
    best_val_loss = float('inf')
    best_epoch = None
    bad_epochs = 0
    history = []

    for epoch in range(1, config.num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, loss_mode)
        val_loss = evaluate_loss(model, val_loader, device, loss_mode)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        print(
            f'{config.experiment_name} | {run_id} | epoch {epoch:02d} '
            f'| train_loss={train_loss:.4f} | val_loss={val_loss:.4f}'
        )

        # Save the best model according to validation loss.
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch = epoch
            bad_epochs = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            bad_epochs += 1
            if bad_epochs >= config.patience:
                print(f'Early stopping at epoch {epoch}; best epoch was {best_epoch}.')
                break

    # Reload the best checkpoint before computing validation/test predictions.
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    val_probs, val_labels = get_predictions(model, val_loader, device)
    test_probs, test_labels = get_predictions(model, test_loader, device)

    # Soft-label metrics are the primary test metrics.
    soft_metrics = soft_label_metrics(test_labels, test_probs)

    # Binary metrics are secondary. We report three human-label definitions:
    # weak presence, exact majority, and strong presence.
    binary_scheme_metrics = {}
    threshold_tables = []
    count_tables = []
    average_precision_tables = []
    tuned_predictions_by_scheme = {}

    for scheme_name, scheme_description in BINARY_TARGET_SCHEMES.items():
        val_binary = binary_target_matrix(val_df, scheme_name)
        test_binary = binary_target_matrix(test_df, scheme_name)

        thresholds, threshold_df = tune_thresholds(val_binary, val_probs)
        threshold_df.insert(0, 'scheme', scheme_name)
        threshold_df.insert(1, 'scheme_description', scheme_description)
        threshold_tables.append(threshold_df)

        fixed_metrics, fixed_ap, fixed_counts, fixed_report, fixed_pred = evaluate_multilabel(
            test_binary,
            test_probs,
            {label: 0.5 for label in LABEL_COLS},
        )
        tuned_metrics, tuned_ap, tuned_counts, tuned_report, tuned_pred = evaluate_multilabel(
            test_binary,
            test_probs,
            thresholds,
        )

        fixed_ap.insert(0, 'scheme', scheme_name)
        tuned_ap.insert(0, 'scheme', scheme_name)
        fixed_ap.insert(1, 'threshold_type', 'fixed_0_5')
        tuned_ap.insert(1, 'threshold_type', 'validation_tuned')
        average_precision_tables.extend([fixed_ap, tuned_ap])

        fixed_counts.insert(0, 'scheme', scheme_name)
        tuned_counts.insert(0, 'scheme', scheme_name)
        fixed_counts.insert(1, 'threshold_type', 'fixed_0_5')
        tuned_counts.insert(1, 'threshold_type', 'validation_tuned')
        count_tables.extend([fixed_counts, tuned_counts])

        tuned_predictions_by_scheme[scheme_name] = tuned_pred
        binary_scheme_metrics[scheme_name] = {
            'description': scheme_description,
            'fixed_0_5_binary_metrics': fixed_metrics,
            'tuned_binary_metrics': tuned_metrics,
            'thresholds': thresholds,
            'test_average_precision_fixed_0_5': fixed_ap.to_dict(orient='records'),
            'test_average_precision_tuned': tuned_ap.to_dict(orient='records'),
            'test_prediction_counts_fixed_0_5': fixed_counts.to_dict(orient='records'),
            'test_prediction_counts_tuned': tuned_counts.to_dict(orient='records'),
            'fixed_classification_report': fixed_report,
            'tuned_classification_report': tuned_report,
        }

    # Save row-level test predictions for error analysis.
    pred_df = test_df[['key', 'SITUATION', 'REAPPRAISAL']].copy()
    for i, label in enumerate(LABEL_COLS):
        pred_df[f'{label}_true_score'] = test_labels[:, i]
        pred_df[f'{label}_score'] = test_probs[:, i]
        for scheme_name in BINARY_TARGET_SCHEMES:
            pred_df[f'{label}_{scheme_name}_true_binary'] = binary_target_matrix(test_df, scheme_name)[:, i]
            pred_df[f'{label}_{scheme_name}_pred_binary'] = tuned_predictions_by_scheme[scheme_name][:, i]

    prefix = run_dir / f'{config.experiment_name}_{run_id}'
    pd.DataFrame(history).to_csv(prefix.with_name(prefix.name + '_training_history.csv'), index=False)
    pred_df.to_csv(prefix.with_name(prefix.name + '_test_predictions.csv'), index=False)
    pd.concat(threshold_tables, ignore_index=True).to_csv(
        prefix.with_name(prefix.name + '_threshold_tuning.csv'), index=False
    )
    soft_metrics.to_csv(prefix.with_name(prefix.name + '_soft_metrics.csv'), index=False)
    pd.concat(count_tables, ignore_index=True).to_csv(
        prefix.with_name(prefix.name + '_prediction_counts.csv'), index=False
    )
    pd.concat(average_precision_tables, ignore_index=True).to_csv(
        prefix.with_name(prefix.name + '_average_precision.csv'), index=False
    )

    summary = {
        'experiment_name': config.experiment_name,
        'model_name': config.model_name,
        'run_id': run_id,
        'loss_mode': loss_mode,
        'binary_target_schemes': BINARY_TARGET_SCHEMES,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'checkpoint_path': str(checkpoint_path),
        'config': asdict(config),
        'soft_summary': macro_soft_summary(soft_metrics),
        'binary_scheme_metrics': binary_scheme_metrics,
        # Majority is kept as top-level convenience because it is the main binary definition.
        'fixed_0_5_binary_metrics': binary_scheme_metrics['majority']['fixed_0_5_binary_metrics'],
        'tuned_binary_metrics': binary_scheme_metrics['majority']['tuned_binary_metrics'],
        'thresholds': binary_scheme_metrics['majority']['thresholds'],
        'test_soft_metrics': soft_metrics.to_dict(orient='records'),
    }
    write_json(prefix.with_name(prefix.name + '_metrics.json'), summary)
    return summary


## Summarize Multiple Runs

These functions combine repeated-seed or leave-one-vignette runs into one table. The grouped summary reports the mean and standard deviation by model.

In [ ]:
RUN_LEAVE_ONE_VIGNETTE = False

if RUN_LEAVE_ONE_VIGNETTE:
    vignette_summary = run_leave_one_vignette_suite(
        model_configs=MODEL_CONFIGS_TO_RUN,
        seed=42,
        loss_mode='binomial_nll',
    )
    display(vignette_summary)


In [ ]:
def summarize_runs(results: list[dict], output_dir: Path = OUTPUT_DIR, name: str = 'summary') -> pd.DataFrame:


    rows = []
    for result in results:
        row = {
            'experiment_name': result['experiment_name'],
            'model_name': result['model_name'],
            'run_id': result['run_id'],
            'loss_mode': result['loss_mode'],
            'best_epoch': result['best_epoch'],
            'best_val_loss': result['best_val_loss'],
        }
        row.update(result['soft_summary'])
        for scheme_name, scheme_payload in result['binary_scheme_metrics'].items():
            row.update({
                f'{scheme_name}_fixed_{k}': v
                for k, v in scheme_payload['fixed_0_5_binary_metrics'].items()
            })
            row.update({
                f'{scheme_name}_tuned_{k}': v
                for k, v in scheme_payload['tuned_binary_metrics'].items()
            })
        rows.append(row)

    summary_df = pd.DataFrame(rows)
    summary_dir = output_dir / 'summaries'
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_dir / f'{name}.csv', index=False)

    grouped = (
        summary_df.groupby(['experiment_name', 'model_name', 'loss_mode'], as_index=False)
        .agg(
            macro_mae_mean=('macro_mae', 'mean'),
            macro_mae_sd=('macro_mae', 'std'),
            mean_pearson_mean=('mean_pearson', 'mean'),
            mean_pearson_sd=('mean_pearson', 'std'),
            weak_presence_tuned_macro_f1_mean=('weak_presence_tuned_macro_f1', 'mean'),
            weak_presence_tuned_macro_f1_sd=('weak_presence_tuned_macro_f1', 'std'),
            majority_tuned_macro_f1_mean=('majority_tuned_macro_f1', 'mean'),
            majority_tuned_macro_f1_sd=('majority_tuned_macro_f1', 'std'),
            strong_presence_tuned_macro_f1_mean=('strong_presence_tuned_macro_f1', 'mean'),
            strong_presence_tuned_macro_f1_sd=('strong_presence_tuned_macro_f1', 'std'),
            n_runs=('run_id', 'count'),
        )
    )
    grouped.to_csv(summary_dir / f'{name}_by_model.csv', index=False)
    return summary_df


def run_random_split_suite(
    model_configs: list[ModelConfig],
    seeds: Iterable[int],
    loss_mode: str = 'binomial_nll',
    split_scheme: str = 'majority',
    output_dir: Path = OUTPUT_DIR,
    summary_name: str | None = None,
    skip_completed: bool = True,
) -> pd.DataFrame:


    results = []
    device = get_device()

    for seed in seeds:
        set_seed(seed)
        train_df, val_df, test_df = make_random_split(dataset, seed, split_scheme=split_scheme)

        for config in model_configs:
            set_seed(seed)
            run_id = f'random_seed{seed}'
            existing_result = load_existing_result(config, run_id, output_dir) if skip_completed else None
            if existing_result is not None:
                print(f'Skipping completed run: {config.experiment_name} | {run_id}')
                results.append(existing_result)
                continue

            results.append(
                run_single_experiment(
                    config,
                    train_df,
                    val_df,
                    test_df,
                    run_id=run_id,
                    output_dir=output_dir,
                    loss_mode=loss_mode,
                    device=device,
                )
            )

    if summary_name is None:
        summary_name = f'random_splits_{loss_mode}'
    return summarize_runs(results, output_dir, name=summary_name)


def run_leave_one_vignette_suite(
    model_configs: list[ModelConfig],
    seed: int = 42,
    loss_mode: str = 'binomial_nll',
    split_scheme: str = 'majority',
    output_dir: Path = OUTPUT_DIR,
) -> pd.DataFrame:
    """Run each selected model with each of the six situations held out as test data."""

    results = []
    device = get_device()

    for vignette_idx, (situation, train_df, val_df, test_df) in enumerate(
        make_leave_one_vignette_splits(dataset, seed, split_scheme=split_scheme)
    ):
        for config in model_configs:
            set_seed(seed)
            run_id = f'leave_vignette_{vignette_idx}_seed{seed}'
            results.append(
                run_single_experiment(
                    config,
                    train_df,
                    val_df,
                    test_df,
                    run_id=run_id,
                    output_dir=output_dir,
                    loss_mode=loss_mode,
                    device=device,
                )
            )

    return summarize_runs(results, output_dir, name=f'leave_one_vignette_{loss_mode}')


## Quick Smoke Test

Use this before a full run.

Set `SMOKE_TEST = True` to train BERT-base for one epoch on one split. This checks that the tokenizer, model loading, dataloaders, loss, and saving logic all work.

In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    smoke_config = ModelConfig(
        experiment_name='ModernBERT_uncased_smoke',
        model_name='modernbert_base',
        batch_size=8,
        learning_rate=1e-5,
        dropout=0.4,
        num_epochs=1,
        patience=1,
        local_files_only=True,
    )
    train_df, val_df, test_df = make_random_split(dataset, seed=42)
    smoke_result = run_single_experiment(
        smoke_config,
        train_df,
        val_df,
        test_df,
        run_id='smoke_seed42',
        loss_mode='binomial_nll',
    )
    display(smoke_result['soft_summary'])


## Repeated Random Splits

This is the main model-comparison block.

By default it runs:

```text
3 models x 5 seeds = 15 trainings
```

For a quick first pass, use the commented two-line quick setting.

In [ ]:
# Safe default: Run All loads existing results but does not train or overwrite models.
RUN_BASELINE_EXPERIMENTS = False
SKIP_COMPLETED_BASELINE = True

SEEDS = [13]
MODEL_CONFIGS_TO_RUN = MODEL_CONFIGS
SUMMARY_NAME = f'random_splits_binomial_nll_seed{SEEDS[0]}' if len(SEEDS) == 1 else 'random_splits_binomial_nll_multi_seed'

if RUN_BASELINE_EXPERIMENTS:
    random_summary = run_random_split_suite(
        model_configs=MODEL_CONFIGS_TO_RUN,
        seeds=SEEDS,
        loss_mode='binomial_nll',
        summary_name=SUMMARY_NAME,
        skip_completed=SKIP_COMPLETED_BASELINE,
    )
else:
    existing_summary_path = SUMMARY_DIR / f'{SUMMARY_NAME}.csv'
    random_summary = pd.read_csv(existing_summary_path) if existing_summary_path.exists() else pd.DataFrame()
    print('Baseline training is disabled. Set RUN_BASELINE_EXPERIMENTS = True to opt in.')
    if not random_summary.empty:
        print('Loaded existing summary:', existing_summary_path)

display(random_summary)


## Optional Leave-One-Vignette-Out Robustness Check

This is slower, but methodologically important.

It asks: if the model never sees one situation during training, can it still detect the reappraisal strategies in that held-out situation?

## Compare Aggregated Results

After the repeated random split run finishes, this cell loads the model-level aggregate summary.

In [ ]:
summary_name = globals().get('SUMMARY_NAME', 'random_splits_binomial_nll')
summary_path = SUMMARY_DIR / f'{summary_name}_by_model.csv'
if summary_path.exists():
    display(
        pd.read_csv(summary_path)
        .sort_values(['macro_mae_mean', 'mean_pearson_mean'], ascending=[True, False])
    )
else:
    print('No aggregate summary yet:', summary_path)


## Results Tables And Matplotlib Figures

These cells read the saved result CSV files and show the main results as simple pandas tables and matplotlib figures.

The soft-label metrics are the primary results because the models were trained to predict annotator-agreement scores. The binary F1 results are secondary interpretability checks.


In [ ]:
# Load saved result files for the result summary section.
import os

MPL_CACHE_DIR = PROJECT_DIR / '.matplotlib_cache'
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support

RESULTS_DIR = OUTPUT_DIR

MODEL_DISPLAY_NAMES = {
    'bert_base_uncased': 'BERT-base',
    'bert_large_uncased': 'BERT-large',
    'modernbert_base': 'ModernBERT-base',
}

MODEL_COLORS = {
    'BERT-base': '#3b82f6',
    'BERT-large': '#0f766e',
    'ModernBERT-base': '#f59e0b',
}

plt.rcParams.update({
    'figure.figsize': (8, 4.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

summary_name = globals().get('SUMMARY_NAME', 'random_splits_binomial_nll')
summary_df = pd.read_csv(SUMMARY_DIR / f'{summary_name}.csv')
summary_df['Model'] = summary_df['experiment_name'].map(MODEL_DISPLAY_NAMES)

summary_by_model_df = pd.read_csv(SUMMARY_DIR / f'{summary_name}_by_model.csv')
summary_by_model_df['Model'] = summary_by_model_df['experiment_name'].map(MODEL_DISPLAY_NAMES)

display(summary_df[['Model', 'run_id', 'loss_mode']].sort_values('Model'))


### Main Model Comparison


In [ ]:
# Table 1: Main model comparison.
# Lower validation loss, MAE, and RMSE are better.
# Higher correlations and F1 are better.
main_results_table = (
    summary_df[[
        'Model',
        'best_epoch',
        'best_val_loss',
        'macro_mae',
        'macro_rmse',
        'mean_pearson',
        'mean_spearman',
        'majority_tuned_macro_f1',
        'strong_presence_tuned_macro_f1',
    ]]
    .rename(columns={
        'best_epoch': 'Best epoch',
        'best_val_loss': 'Validation loss',
        'macro_mae': 'Macro MAE',
        'macro_rmse': 'Macro RMSE',
        'mean_pearson': 'Mean Pearson',
        'mean_spearman': 'Mean Spearman',
        'majority_tuned_macro_f1': 'Majority Macro F1',
        'strong_presence_tuned_macro_f1': 'Strong Presence Macro F1',
    })
    .sort_values('Macro MAE')
    .reset_index(drop=True)
)

display(main_results_table.round(3))


In [ ]:
# Figure 1: Macro MAE by model.
# Lower is better.
plot_df = main_results_table.sort_values('Macro MAE')

fig, ax = plt.subplots(figsize=(7, 4))
colors = [MODEL_COLORS[model] for model in plot_df['Model']]
bars = ax.bar(plot_df['Model'], plot_df['Macro MAE'], color=colors)

ax.set_title('Macro MAE by Model')
ax.set_ylabel('Macro MAE')
ax.set_xlabel('Model')
ax.set_ylim(0, plot_df['Macro MAE'].max() * 1.25)

for bar in bars:
    value = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


### Actual Strategy Prevalence At 0.6 Threshold


In [ ]:
# Table 2: Actual number of examples where each strategy is present.
# Here, 'present' means the true soft label is at least 0.6.
# With five annotators, this roughly means at least 3 out of 5 annotators marked the strategy present.
PRESENCE_THRESHOLD = 0.6

if 'dataset' not in globals():
    dataset = load_reappraisal_dataset()

actual_strategy_counts = pd.DataFrame({
    'Label': LABEL_COLS,
    'Strategy': [LABEL_NAMES[label] for label in LABEL_COLS],
    'Actual present count': [(dataset[label] >= PRESENCE_THRESHOLD).sum() for label in LABEL_COLS],
    'Total examples': len(dataset),
})
actual_strategy_counts['Actual present percent'] = (
    actual_strategy_counts['Actual present count'] / actual_strategy_counts['Total examples'] * 100
)
actual_strategy_counts = actual_strategy_counts.sort_values('Actual present count', ascending=False).reset_index(drop=True)

display(actual_strategy_counts.round(2))


In [ ]:
# Figure 2: Actual number of present examples per strategy at the 0.6 threshold.
plot_df = actual_strategy_counts.sort_values('Actual present count')

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(plot_df['Label'], plot_df['Actual present count'], color='#0f766e')

ax.set_title('Actual Strategy Presence in the Dataset, Threshold >= 0.6')
ax.set_xlabel('Number of examples')
ax.set_ylabel('Strategy')

for bar in bars:
    value = int(bar.get_width())
    ax.text(value + 3, bar.get_y() + bar.get_height() / 2, str(value), va='center')

plt.tight_layout()
plt.show()
